# RQ5 — Strategy Robustness, Profit Concentration and Multi-Contract Scaling

This notebook adds depth to the RQ5 options analysis without replacing the original primary backtest.

It answers five practical questions:

1. **Are the profits dependent on only one or two extreme winning trades?**
2. **What happens after a contract first touches a −50% stop level?**
3. **What happens after a contract first reaches +100%?**
4. **When during the final hour is the strategy's value created?**
5. **Would scaling to multiple contracts and taking partial profits improve the payoff profile?**

It also includes:

- Call versus Put decomposition;
- transaction-cost break-even sensitivity;
- same-date random-direction Monte Carlo;
- optional random-day control retrieval guidance;
- hypothetical-account capital exposure;
- maximum-adverse-excursion versus maximum-favourable-excursion charts.

## Primary multi-contract sensitivity rule

The first multi-contract rule is fixed before reviewing its result:

> **Buy 2 identical contracts. Use a −50% hard stop on both. If the option reaches +100%, sell 1 contract and keep 1 runner until the normal end-of-hour exit.**

This is called:

`2C_TP100_RUNNER_CLOSE`

The purpose is to test whether partial profit-taking can lock in some gain while preserving exposure to the very large winners observed in the earlier RQ5 analysis.

## Additional multi-contract sensitivity rules

These are explanatory sensitivity tests, not newly optimised primary strategies:

- `2C_TP100_RUNNER_BE`
  - buy 2;
  - initial −50% stop;
  - sell 1 at +100%;
  - move the runner stop to break-even from the next minute.

- `2C_TP100_RUNNER_TRAIL25`
  - buy 2;
  - initial −50% stop;
  - sell 1 at +100%;
  - from the next minute, trail the runner 25% below its running high.

- `3C_TP50_TP100_RUNNER_CLOSE`
  - buy 3;
  - initial −50% stop;
  - sell 1 at +50%;
  - sell 1 at +100%;
  - keep 1 runner to the normal exit.

ATM remains the primary strike specification. OTM results remain sensitivity analyses.

## Important limitations

- The historical sample contains few trade opportunities.
- Multi-contract rules increase capital at risk and should not be described as suitable for a small account merely because they improve historical P&L.
- One-minute OHLC bars do not reveal intraminute price order.
- Historical NBBO bid/ask quotes are unavailable under the Options Starter dataset.
- Any newly interesting runner configuration found here must be treated as exploratory unless frozen before a future untouched holdout.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import json
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42

PRIMARY_STRIKE_OFFSET = 0
STRIKE_OFFSETS_POINTS = [0, 5, 10, 15, 20, 25, 30]
PRIMARY_COST_SCENARIO = "medium_cost"

ENTRY_WINDOW_START = "15:00"
ENTRY_WINDOW_END = "15:05"
CLOSE_EXIT_WINDOW_START = "15:55"
CLOSE_EXIT_WINDOW_END = "15:59"

DEFAULT_CONTRACT_MULTIPLIER = 100.0

# Same execution assumptions as the original RQ5 notebooks.
EXECUTION_SCENARIOS = {
    "frictionless": {
        "min_penalty_points": 0.00,
        "pct_of_reference_price": 0.000,
        "commission_per_side_usd": 0.00,
    },
    "low_cost": {
        "min_penalty_points": 0.05,
        "pct_of_reference_price": 0.020,
        "commission_per_side_usd": 1.00,
    },
    "medium_cost": {
        "min_penalty_points": 0.10,
        "pct_of_reference_price": 0.050,
        "commission_per_side_usd": 1.50,
    },
    "severe_cost": {
        "min_penalty_points": 0.20,
        "pct_of_reference_price": 0.100,
        "commission_per_side_usd": 2.00,
    },
}

# Checkpoints for "when is value created?" analysis.
TIME_CHECKPOINTS = [
    "15:05",
    "15:10",
    "15:15",
    "15:20",
    "15:30",
    "15:40",
    "15:45",
    "15:50",
    "15:55",
    "15:59",
]

# Hypothetical account sizes are descriptive stress tests only.
HYPOTHETICAL_ACCOUNTS_USD = [2_500, 5_000, 10_000, 25_000]

# Same-date random-direction Monte Carlo.
N_RANDOM_DIRECTION_SIMULATIONS = 10_000

# Profit concentration.
TOP_WINNER_COUNTS = [1, 2, 3, 5]

# Break-even execution-cost grid.
BREAK_EVEN_PCT_GRID = np.arange(0.00, 0.305, 0.005)
BREAK_EVEN_MIN_PENALTY_POINTS = 0.00
BREAK_EVEN_COMMISSION_PER_SIDE_USD = 1.50

# Multi-contract rule definitions.
MULTI_CONTRACT_RULES = {
    "1C_HOLD_CLOSE": {
        "contracts": 1,
        "initial_stop_pct": None,
        "tp1_pct": None,
        "tp1_contracts": 0,
        "tp2_pct": None,
        "tp2_contracts": 0,
        "runner_mode": "close",
    },
    "1C_STOP50_HOLD_CLOSE": {
        "contracts": 1,
        "initial_stop_pct": -0.50,
        "tp1_pct": None,
        "tp1_contracts": 0,
        "tp2_pct": None,
        "tp2_contracts": 0,
        "runner_mode": "close",
    },
    # Primary multi-contract sensitivity.
    "2C_TP100_RUNNER_CLOSE": {
        "contracts": 2,
        "initial_stop_pct": -0.50,
        "tp1_pct": 1.00,
        "tp1_contracts": 1,
        "tp2_pct": None,
        "tp2_contracts": 0,
        "runner_mode": "close",
    },
    "2C_TP100_RUNNER_BE": {
        "contracts": 2,
        "initial_stop_pct": -0.50,
        "tp1_pct": 1.00,
        "tp1_contracts": 1,
        "tp2_pct": None,
        "tp2_contracts": 0,
        "runner_mode": "breakeven_after_tp1",
    },
    "2C_TP100_RUNNER_TRAIL25": {
        "contracts": 2,
        "initial_stop_pct": -0.50,
        "tp1_pct": 1.00,
        "tp1_contracts": 1,
        "tp2_pct": None,
        "tp2_contracts": 0,
        "runner_mode": "trail_after_tp1",
        "trail_pct": 0.25,
    },
    "3C_TP50_TP100_RUNNER_CLOSE": {
        "contracts": 3,
        "initial_stop_pct": -0.50,
        "tp1_pct": 0.50,
        "tp1_contracts": 1,
        "tp2_pct": 1.00,
        "tp2_contracts": 1,
        "runner_mode": "close",
    },
}

PRIMARY_MULTI_CONTRACT_RULE = "2C_TP100_RUNNER_CLOSE"

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 280)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 2. Locate and load existing RQ5 outputs

In [ ]:
def locate_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "market.duckdb").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find data/market.duckdb. "
        "Run this notebook inside the dissertation project."
    )


PROJECT_ROOT = locate_project_root(Path.cwd())

RQ5_ROOT = PROJECT_ROOT / "outputs" / "rq5_options_trading"
RAW_ROOT = RQ5_ROOT / "raw"
TABLE_ROOT = RQ5_ROOT / "tables"
FIGURE_ROOT = RQ5_ROOT / "figures"
ROBUST_ROOT = RQ5_ROOT / "strategy_robustness"

ROBUST_ROOT.mkdir(parents=True, exist_ok=True)
TABLE_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

BARS_PATH = RAW_ROOT / "rq5_option_minute_bars.parquet"
SELECTION_PATH = TABLE_ROOT / "rq5_contract_selection.csv"
CANDIDATES_PATH = TABLE_ROOT / "rq5_primary_candidate_sessions.csv"
TRADE_LOG_PATH = RQ5_ROOT / "backtest" / "rq5_complete_trade_log.parquet"

required = [
    BARS_PATH,
    SELECTION_PATH,
    CANDIDATES_PATH,
    TRADE_LOG_PATH,
]
missing = [str(p) for p in required if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Required RQ5 files are missing.\n- " + "\n- ".join(missing)
    )

bars = pd.read_parquet(BARS_PATH)
selection = pd.read_csv(SELECTION_PATH)
candidates = pd.read_csv(CANDIDATES_PATH)
trade_log = pd.read_parquet(TRADE_LOG_PATH)

bars["session_date"] = pd.to_datetime(bars["session_date"])
bars["timestamp"] = pd.to_datetime(bars["timestamp"])
selection["session_date"] = pd.to_datetime(selection["session_date"])
candidates["session_date"] = pd.to_datetime(candidates["session_date"])
trade_log["session_date"] = pd.to_datetime(trade_log["session_date"])

bars_unique = (
    bars.sort_values(["session_date", "option_ticker", "timestamp"])
    .drop_duplicates(
        ["session_date", "option_ticker", "timestamp"],
        keep="first",
    )
    .reset_index(drop=True)
)

selection = selection[
    selection["selection_status"].eq("selected")
].copy()

selection["otm_offset_points"] = pd.to_numeric(
    selection["otm_offset_points"],
    errors="coerce",
)

candidates["ml_direction"] = np.where(
    candidates["rq1_prediction"].astype(int).eq(1),
    "call",
    "put",
)

candidates["mean_reversion_direction"] = np.where(
    candidates["ret_last_60m"] < 0,
    "call",
    "put",
)

primary_trades = trade_log[
    trade_log["strategy"].eq("RQ1_ML")
    & trade_log["otm_offset_points"].eq(PRIMARY_STRIKE_OFFSET)
    & trade_log["execution_scenario"].eq(PRIMARY_COST_SCENARIO)
].copy()

primary_trades = primary_trades.sort_values("session_date").reset_index(drop=True)

print("Primary ML ATM trades:", len(primary_trades))
print("Candidate dates:", len(candidates))
print("Unique option bars:", len(bars_unique))

## 3. Utility functions

In [ ]:
def execution_penalty(reference_price, params):
    return max(
        float(params["min_penalty_points"]),
        float(params["pct_of_reference_price"]) * float(reference_price),
    )


def execute_buy(reference_price, params):
    penalty = execution_penalty(reference_price, params)
    return float(reference_price) + penalty


def execute_sell(reference_price, params):
    reference_price = max(0.0, float(reference_price))
    penalty = execution_penalty(reference_price, params)
    return max(0.0, reference_price - penalty)


def first_entry_bar(contract_bars):
    work = contract_bars.sort_values("timestamp").copy()
    hhmm = work["timestamp"].dt.strftime("%H:%M")

    eligible = work[
        (hhmm >= ENTRY_WINDOW_START)
        & (hhmm <= ENTRY_WINDOW_END)
    ]

    if eligible.empty:
        return None

    return eligible.iloc[0]


def last_close_bar(contract_bars):
    work = contract_bars.sort_values("timestamp").copy()
    hhmm = work["timestamp"].dt.strftime("%H:%M")

    eligible = work[
        (hhmm >= CLOSE_EXIT_WINDOW_START)
        & (hhmm <= CLOSE_EXIT_WINDOW_END)
    ]

    if eligible.empty:
        return None

    return eligible.iloc[-1]


def price_at_checkpoint(contract_bars, entry_ts, checkpoint):
    work = contract_bars[
        contract_bars["timestamp"] >= entry_ts
    ].sort_values("timestamp").copy()

    if work.empty:
        return None

    tz = pd.Timestamp(entry_ts).tz
    target = pd.Timestamp(
        f"{pd.Timestamp(entry_ts).date()} {checkpoint}",
        tz=tz,
    )
    lower = target - pd.Timedelta(minutes=5)

    eligible = work[
        (work["timestamp"] >= lower)
        & (work["timestamp"] <= target)
    ]

    if eligible.empty:
        return None

    return eligible.iloc[-1]


def get_selected_contract(date, option_type, offset):
    rows = selection[
        selection["session_date"].eq(pd.Timestamp(date))
        & selection["option_type"].str.lower().eq(option_type)
        & selection["otm_offset_points"].eq(offset)
    ]

    if rows.empty:
        return None

    return rows.iloc[0]


def get_contract_bars(date, ticker):
    return bars_unique[
        bars_unique["session_date"].eq(pd.Timestamp(date))
        & bars_unique["option_ticker"].eq(ticker)
    ].sort_values("timestamp").copy()


def profit_factor(pnl):
    pnl = pd.to_numeric(pnl, errors="coerce").dropna()
    gains = pnl[pnl > 0].sum()
    losses = -pnl[pnl < 0].sum()

    if losses == 0:
        return np.inf if gains > 0 else np.nan

    return float(gains / losses)


def max_drawdown_from_pnl(frame, pnl_col="net_pnl_usd"):
    ordered = frame.sort_values("session_date")
    equity = ordered[pnl_col].cumsum()
    peak = equity.cummax()
    dd = equity - peak
    return float(dd.min()) if len(dd) else np.nan

# Part A — Profit concentration and dependence on extreme winners

## 4. How concentrated is total P&L?

In [ ]:
concentration_source = primary_trades[
    [
        "session_date",
        "net_pnl_usd",
        "net_return_on_premium",
        "required_option_type",
    ]
].copy()

total_pnl = concentration_source["net_pnl_usd"].sum()

sorted_winners = concentration_source.sort_values(
    "net_pnl_usd",
    ascending=False,
).reset_index(drop=True)

concentration_rows = [
    {
        "scenario": "All trades",
        "removed_top_winners": 0,
        "remaining_trades": len(sorted_winners),
        "total_pnl_usd": total_pnl,
        "mean_pnl_usd": sorted_winners["net_pnl_usd"].mean(),
        "win_rate": (sorted_winners["net_pnl_usd"] > 0).mean(),
    }
]

for n in TOP_WINNER_COUNTS:
    if n >= len(sorted_winners):
        continue

    remaining = sorted_winners.iloc[n:].copy()

    concentration_rows.append(
        {
            "scenario": f"Remove best {n}",
            "removed_top_winners": n,
            "remaining_trades": len(remaining),
            "total_pnl_usd": remaining["net_pnl_usd"].sum(),
            "mean_pnl_usd": remaining["net_pnl_usd"].mean(),
            "win_rate": (remaining["net_pnl_usd"] > 0).mean(),
        }
    )

profit_concentration = pd.DataFrame(concentration_rows)

top_contribution_rows = []
positive_total = sorted_winners.loc[
    sorted_winners["net_pnl_usd"] > 0,
    "net_pnl_usd",
].sum()

for n in TOP_WINNER_COUNTS:
    top_sum = sorted_winners.head(n)["net_pnl_usd"].sum()

    top_contribution_rows.append(
        {
            "top_n_trades": n,
            "top_n_pnl_usd": top_sum,
            "share_of_net_total_pnl": (
                top_sum / total_pnl
                if total_pnl != 0
                else np.nan
            ),
            "share_of_total_positive_pnl": (
                top_sum / positive_total
                if positive_total > 0
                else np.nan
            ),
        }
    )

top_contribution = pd.DataFrame(top_contribution_rows)

display(profit_concentration)
display(top_contribution)

profit_concentration.to_csv(
    TABLE_ROOT / "rq5_profit_concentration.csv",
    index=False,
)
top_contribution.to_csv(
    TABLE_ROOT / "rq5_top_winner_contribution.csv",
    index=False,
)

## 5. Leave-one-trade-out profitability

In [ ]:
leave_one_out_rows = []

for idx, row in concentration_source.iterrows():
    remaining = concentration_source.drop(index=idx)

    leave_one_out_rows.append(
        {
            "removed_session": row["session_date"],
            "removed_trade_pnl_usd": row["net_pnl_usd"],
            "remaining_total_pnl_usd": remaining["net_pnl_usd"].sum(),
            "remaining_mean_pnl_usd": remaining["net_pnl_usd"].mean(),
            "remaining_profitable": remaining["net_pnl_usd"].sum() > 0,
        }
    )

leave_one_out = pd.DataFrame(leave_one_out_rows).sort_values(
    "remaining_total_pnl_usd"
)

display(leave_one_out)

leave_one_out.to_csv(
    TABLE_ROOT / "rq5_leave_one_trade_out.csv",
    index=False,
)

# Part B — What happens after stop and take-profit thresholds are touched?

## 6. Build minute-path diagnostics for each primary ATM ML trade

In [ ]:
path_rows = []

scenario_params = EXECUTION_SCENARIOS[PRIMARY_COST_SCENARIO]

for trade in primary_trades.itertuples(index=False):
    date = pd.Timestamp(trade.session_date)
    ticker = trade.option_ticker

    contract_bars = get_contract_bars(date, ticker)

    entry_bar = first_entry_bar(contract_bars)
    close_bar = last_close_bar(contract_bars)

    if entry_bar is None or close_bar is None:
        continue

    entry_ref = float(entry_bar["open"])
    entry_exec = execute_buy(entry_ref, scenario_params)
    entry_ts = pd.Timestamp(entry_bar["timestamp"])
    close_ts = pd.Timestamp(close_bar["timestamp"])

    path = contract_bars[
        (contract_bars["timestamp"] >= entry_ts)
        & (contract_bars["timestamp"] <= close_ts)
    ].sort_values("timestamp").copy()

    if path.empty:
        continue

    stop_level = entry_exec * 0.50
    tp50_level = entry_exec * 1.50
    tp100_level = entry_exec * 2.00

    stop_hits = path[path["low"] <= stop_level]
    tp50_hits = path[path["high"] >= tp50_level]
    tp100_hits = path[path["high"] >= tp100_level]

    stop_ts = (
        pd.Timestamp(stop_hits.iloc[0]["timestamp"])
        if len(stop_hits)
        else pd.NaT
    )

    tp50_ts = (
        pd.Timestamp(tp50_hits.iloc[0]["timestamp"])
        if len(tp50_hits)
        else pd.NaT
    )

    tp100_ts = (
        pd.Timestamp(tp100_hits.iloc[0]["timestamp"])
        if len(tp100_hits)
        else pd.NaT
    )

    after_stop = (
        path[path["timestamp"] > stop_ts].copy()
        if pd.notna(stop_ts)
        else pd.DataFrame()
    )

    after_tp100 = (
        path[path["timestamp"] > tp100_ts].copy()
        if pd.notna(tp100_ts)
        else pd.DataFrame()
    )

    later_recovered_entry = (
        bool((after_stop["high"] >= entry_exec).any())
        if len(after_stop)
        else False
    )

    later_reached_plus50 = (
        bool((after_stop["high"] >= tp50_level).any())
        if len(after_stop)
        else False
    )

    later_reached_plus100 = (
        bool((after_stop["high"] >= tp100_level).any())
        if len(after_stop)
        else False
    )

    later_max_after_tp100 = (
        float(after_tp100["high"].max())
        if len(after_tp100)
        else np.nan
    )

    later_max_return_after_tp100 = (
        later_max_after_tp100 / entry_exec - 1
        if np.isfinite(later_max_after_tp100)
        else np.nan
    )

    max_high = float(path["high"].max())
    min_low = float(path["low"].min())

    path_rows.append(
        {
            "session_date": date,
            "option_ticker": ticker,
            "entry_execution_price": entry_exec,
            "stop50_touched": len(stop_hits) > 0,
            "stop50_first_touch": stop_ts,
            "tp50_touched": len(tp50_hits) > 0,
            "tp50_first_touch": tp50_ts,
            "tp100_touched": len(tp100_hits) > 0,
            "tp100_first_touch": tp100_ts,
            "stop_before_tp100": (
                pd.notna(stop_ts)
                and (
                    pd.isna(tp100_ts)
                    or stop_ts < tp100_ts
                )
            ),
            "tp100_before_stop": (
                pd.notna(tp100_ts)
                and (
                    pd.isna(stop_ts)
                    or tp100_ts < stop_ts
                )
            ),
            "after_stop_recovered_to_entry": later_recovered_entry,
            "after_stop_reached_plus50": later_reached_plus50,
            "after_stop_reached_plus100": later_reached_plus100,
            "close_profitable_vs_entry": (
                float(close_bar["close"]) > entry_exec
            ),
            "max_favorable_excursion": (
                max_high / entry_exec - 1
            ),
            "max_adverse_excursion": (
                min_low / entry_exec - 1
            ),
            "max_return_after_first_tp100": later_max_return_after_tp100,
            "close_return_reference": (
                float(close_bar["close"]) / entry_exec - 1
            ),
        }
    )

path_diagnostics = pd.DataFrame(path_rows)

display(path_diagnostics)

path_diagnostics.to_csv(
    TABLE_ROOT / "rq5_stop_tp_path_recovery.csv",
    index=False,
)

## 7. Stop-out recovery summary

In [ ]:
stopped = path_diagnostics[
    path_diagnostics["stop50_touched"]
].copy()

stop_recovery_summary = pd.DataFrame(
    {
        "metric": [
            "Trades touching -50%",
            "Later recovered to entry",
            "Later reached +50%",
            "Later reached +100%",
            "Eventually closed above entry",
        ],
        "count": [
            len(stopped),
            int(stopped["after_stop_recovered_to_entry"].sum()),
            int(stopped["after_stop_reached_plus50"].sum()),
            int(stopped["after_stop_reached_plus100"].sum()),
            int(stopped["close_profitable_vs_entry"].sum()),
        ],
    }
)

stop_recovery_summary["share_of_stop_touches"] = (
    stop_recovery_summary["count"] / len(stopped)
    if len(stopped)
    else np.nan
)

display(stop_recovery_summary)

stop_recovery_summary.to_csv(
    TABLE_ROOT / "rq5_stop50_recovery_summary.csv",
    index=False,
)

## 8. What happened after +100% was first reached?

In [ ]:
tp100 = path_diagnostics[
    path_diagnostics["tp100_touched"]
].copy()

if len(tp100):
    post_tp_bins = pd.cut(
        tp100["max_return_after_first_tp100"],
        bins=[-np.inf, 1.5, 2.0, 3.0, np.inf],
        labels=[
            "Did not exceed +150% after first +100%",
            "Later reached +150% to +200%",
            "Later reached +200% to +300%",
            "Later exceeded +300%",
        ],
        right=False,
    )

    post_tp_summary = (
        post_tp_bins.value_counts(sort=False)
        .rename_axis("post_tp100_path")
        .reset_index(name="trades")
    )
    post_tp_summary["share"] = (
        post_tp_summary["trades"] / len(tp100)
    )

    display(post_tp_summary)

    post_tp_summary.to_csv(
        TABLE_ROOT / "rq5_post_tp100_upside_summary.csv",
        index=False,
    )
else:
    print("No primary trades reached +100%.")

# Part C — When during the final hour is value created?

## 9. Time-of-hour option return and P&L development

In [ ]:
time_rows = []

for trade in primary_trades.itertuples(index=False):
    date = pd.Timestamp(trade.session_date)
    ticker = trade.option_ticker

    contract_bars = get_contract_bars(date, ticker)
    entry_bar = first_entry_bar(contract_bars)

    if entry_bar is None:
        continue

    entry_exec = execute_buy(
        float(entry_bar["open"]),
        scenario_params,
    )
    entry_ts = pd.Timestamp(entry_bar["timestamp"])
    multiplier = float(trade.contract_multiplier)

    for checkpoint in TIME_CHECKPOINTS:
        bar = price_at_checkpoint(
            contract_bars,
            entry_ts,
            checkpoint,
        )

        if bar is None:
            continue

        exit_exec = execute_sell(
            float(bar["close"]),
            scenario_params,
        )

        commission = (
            2
            * scenario_params["commission_per_side_usd"]
        )

        pnl = (
            (exit_exec - entry_exec) * multiplier
            - commission
        )

        capital = (
            entry_exec * multiplier
            + scenario_params["commission_per_side_usd"]
        )

        time_rows.append(
            {
                "session_date": date,
                "checkpoint": checkpoint,
                "net_pnl_usd": pnl,
                "return_on_premium": (
                    pnl / capital
                    if capital > 0
                    else np.nan
                ),
                "profitable": pnl > 0,
            }
        )

time_development = pd.DataFrame(time_rows)

time_summary = (
    time_development.groupby("checkpoint")
    .agg(
        trades=("session_date", "size"),
        mean_pnl_usd=("net_pnl_usd", "mean"),
        median_pnl_usd=("net_pnl_usd", "median"),
        total_pnl_usd=("net_pnl_usd", "sum"),
        mean_return_on_premium=(
            "return_on_premium",
            "mean",
        ),
        median_return_on_premium=(
            "return_on_premium",
            "median",
        ),
        win_rate=("profitable", "mean"),
    )
    .reset_index()
)

time_summary["checkpoint_order"] = time_summary[
    "checkpoint"
].apply(TIME_CHECKPOINTS.index)

time_summary = time_summary.sort_values(
    "checkpoint_order"
).drop(columns="checkpoint_order")

display(time_summary)

time_summary.to_csv(
    TABLE_ROOT / "rq5_time_of_hour_development.csv",
    index=False,
)

fig, ax1 = plt.subplots(figsize=(10, 5.5))

x = np.arange(len(time_summary))

ax1.plot(
    x,
    time_summary["mean_return_on_premium"],
    marker="o",
    color="tab:blue",
    label="Mean return on premium",
)
ax1.set_ylabel("Mean return on premium", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.set_xticks(x)
ax1.set_xticklabels(
    time_summary["checkpoint"],
    rotation=45,
)
ax1.set_xlabel("Exit checkpoint")

ax2 = ax1.twinx()
ax2.plot(
    x,
    time_summary["win_rate"],
    marker="s",
    color="tab:orange",
    label="Win rate",
)
ax2.set_ylabel("Win rate", color="tab:orange")
ax2.tick_params(axis="y", labelcolor="tab:orange")

ax1.set_title(
    "RQ5: Development of Option Performance Through the Final Hour"
)
ax1.grid(alpha=0.2)

fig.tight_layout()
time_fig = FIGURE_ROOT / "rq5_time_of_hour_development.png"
fig.savefig(time_fig, dpi=220, bbox_inches="tight")
print("Saved:", time_fig)
plt.show()

# Part D — Call versus Put decomposition

## 10. Is profitability concentrated on one option side?

In [ ]:
side_summary = (
    primary_trades.groupby("required_option_type")
    .agg(
        trades=("session_date", "size"),
        total_pnl_usd=("net_pnl_usd", "sum"),
        mean_pnl_usd=("net_pnl_usd", "mean"),
        median_pnl_usd=("net_pnl_usd", "median"),
        mean_return_on_premium=(
            "net_return_on_premium",
            "mean",
        ),
        win_rate=("winner", "mean"),
        median_capital_at_risk_usd=(
            "capital_at_risk_usd",
            "median",
        ),
    )
    .reset_index()
)

display(side_summary)

side_summary.to_csv(
    TABLE_ROOT / "rq5_call_put_decomposition.csv",
    index=False,
)

# Part E — Same-date random-direction Monte Carlo

## 11. Random-direction benchmark on the same selected dates

This test **does not** ask whether the model selected better dates than random days.

Instead it asks:

> Once the RQ2/RQ3/RQ1-confidence filter has selected these dates, did the RQ1 direction add value beyond randomly choosing Call or Put?

For every candidate date, the notebook uses the actually retrieved ATM Call and ATM Put and simulates 10,000 random direction assignments.

This is possible without collecting any new market days.

A stronger **random-day selection** benchmark requires downloading option data for non-candidate dates and is discussed later.

In [ ]:
def hold_trade_for_direction(date, option_type, offset=0):
    selected = get_selected_contract(
        date,
        option_type,
        offset,
    )

    if selected is None:
        return None

    contract_bars = get_contract_bars(
        date,
        selected["ticker"],
    )

    entry = first_entry_bar(contract_bars)
    exit_ = last_close_bar(contract_bars)

    if entry is None or exit_ is None:
        return None

    entry_exec = execute_buy(
        float(entry["open"]),
        scenario_params,
    )
    exit_exec = execute_sell(
        float(exit_["close"]),
        scenario_params,
    )

    multiplier_raw = pd.to_numeric(
        pd.Series(
            [selected.get("shares_per_contract", np.nan)]
        ),
        errors="coerce",
    ).iloc[0]

    multiplier = (
        float(multiplier_raw)
        if np.isfinite(multiplier_raw)
        and multiplier_raw > 0
        else DEFAULT_CONTRACT_MULTIPLIER
    )

    commission = (
        2
        * scenario_params["commission_per_side_usd"]
    )

    pnl = (
        (exit_exec - entry_exec)
        * multiplier
        - commission
    )

    return pnl


direction_matrix_rows = []

for candidate in candidates.itertuples(index=False):
    date = pd.Timestamp(candidate.session_date)

    call_pnl = hold_trade_for_direction(
        date,
        "call",
        PRIMARY_STRIKE_OFFSET,
    )
    put_pnl = hold_trade_for_direction(
        date,
        "put",
        PRIMARY_STRIKE_OFFSET,
    )

    if call_pnl is None or put_pnl is None:
        continue

    direction_matrix_rows.append(
        {
            "session_date": date,
            "call_pnl_usd": call_pnl,
            "put_pnl_usd": put_pnl,
            "ml_direction": candidate.ml_direction,
        }
    )

direction_matrix = pd.DataFrame(direction_matrix_rows)

actual_ml_total = 0.0

for row in direction_matrix.itertuples(index=False):
    actual_ml_total += (
        row.call_pnl_usd
        if row.ml_direction == "call"
        else row.put_pnl_usd
    )

rng = np.random.default_rng(RANDOM_STATE)

random_totals = np.empty(
    N_RANDOM_DIRECTION_SIMULATIONS,
    dtype=float,
)

for i in range(N_RANDOM_DIRECTION_SIMULATIONS):
    choose_call = rng.integers(
        0,
        2,
        size=len(direction_matrix),
    ).astype(bool)

    pnl = np.where(
        choose_call,
        direction_matrix["call_pnl_usd"].to_numpy(),
        direction_matrix["put_pnl_usd"].to_numpy(),
    )

    random_totals[i] = pnl.sum()

random_percentile = (
    (random_totals < actual_ml_total).mean()
)

random_direction_summary = pd.DataFrame(
    {
        "metric": [
            "Actual ML total P&L",
            "Random-direction mean total P&L",
            "Random-direction median total P&L",
            "Random-direction 5th percentile",
            "Random-direction 95th percentile",
            "Share random simulations below ML",
        ],
        "value": [
            actual_ml_total,
            random_totals.mean(),
            np.median(random_totals),
            np.quantile(random_totals, 0.05),
            np.quantile(random_totals, 0.95),
            random_percentile,
        ],
    }
)

display(random_direction_summary)

random_direction_summary.to_csv(
    TABLE_ROOT / "rq5_random_direction_monte_carlo.csv",
    index=False,
)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.hist(
    random_totals,
    bins=50,
    alpha=0.75,
    color="steelblue",
)
ax.axvline(
    actual_ml_total,
    color="darkred",
    linewidth=2,
    label=f"Actual ML = ${actual_ml_total:,.0f}",
)
ax.set_xlabel("Total P&L across matched candidate dates (USD)")
ax.set_ylabel("Monte Carlo simulations")
ax.set_title(
    "RQ5: Same-Date Random-Direction Monte Carlo"
)
ax.legend()

fig.tight_layout()
random_fig = FIGURE_ROOT / "rq5_random_direction_monte_carlo.png"
fig.savefig(random_fig, dpi=220, bbox_inches="tight")
print("Saved:", random_fig)
plt.show()

## 12. Full random-day benchmark — methodological note

A true test of:

> "Are the RQ1–RQ3 selected days better than randomly chosen days?"

requires option prices on **non-candidate days**.

The current RQ5 retrieval deliberately downloaded contracts only for model-selected candidate dates. Therefore a random-day P&L benchmark cannot be reconstructed honestly from the existing files.

A follow-up retrieval can:

1. draw a reproducible control pool of non-candidate outer-fold dates;
2. retrieve ATM Calls and Puts for those dates;
3. use identical 15:00 entry / end-of-hour exit rules;
4. repeatedly draw matched samples of the same size as the model strategy;
5. compare the observed strategy with the random-day P&L distribution.

Do not substitute random directions on selected dates for this stronger day-selection test. They answer different questions.

# Part F — Break-even transaction-cost analysis

## 13. How much adverse execution can ATM tolerate before total P&L reaches zero?

In [ ]:
break_even_rows = []

# Re-use the same ML ATM candidate dates and contracts, but vary only the
# percentage adverse execution penalty.
for pct_penalty in BREAK_EVEN_PCT_GRID:
    params = {
        "min_penalty_points": BREAK_EVEN_MIN_PENALTY_POINTS,
        "pct_of_reference_price": float(pct_penalty),
        "commission_per_side_usd": (
            BREAK_EVEN_COMMISSION_PER_SIDE_USD
        ),
    }

    pnl_values = []

    for candidate in candidates.itertuples(index=False):
        option_type = candidate.ml_direction

        selected = get_selected_contract(
            candidate.session_date,
            option_type,
            PRIMARY_STRIKE_OFFSET,
        )

        if selected is None:
            continue

        contract_bars = get_contract_bars(
            candidate.session_date,
            selected["ticker"],
        )

        entry = first_entry_bar(contract_bars)
        exit_ = last_close_bar(contract_bars)

        if entry is None or exit_ is None:
            continue

        entry_exec = execute_buy(
            float(entry["open"]),
            params,
        )
        exit_exec = execute_sell(
            float(exit_["close"]),
            params,
        )

        multiplier_raw = pd.to_numeric(
            pd.Series(
                [selected.get("shares_per_contract", np.nan)]
            ),
            errors="coerce",
        ).iloc[0]

        multiplier = (
            float(multiplier_raw)
            if np.isfinite(multiplier_raw)
            and multiplier_raw > 0
            else DEFAULT_CONTRACT_MULTIPLIER
        )

        commission = (
            2
            * params["commission_per_side_usd"]
        )

        pnl_values.append(
            (exit_exec - entry_exec)
            * multiplier
            - commission
        )

    break_even_rows.append(
        {
            "pct_adverse_execution_per_side": pct_penalty,
            "trades": len(pnl_values),
            "total_pnl_usd": np.sum(pnl_values),
            "mean_pnl_usd": np.mean(pnl_values)
            if pnl_values
            else np.nan,
        }
    )

break_even_cost = pd.DataFrame(break_even_rows)

non_positive = break_even_cost[
    break_even_cost["total_pnl_usd"] <= 0
]

estimated_break_even_pct = (
    non_positive.iloc[0][
        "pct_adverse_execution_per_side"
    ]
    if len(non_positive)
    else np.nan
)

print(
    "First tested adverse execution percentage with non-positive total P&L:",
    estimated_break_even_pct,
)

display(break_even_cost)

break_even_cost.to_csv(
    TABLE_ROOT / "rq5_break_even_execution_cost.csv",
    index=False,
)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(
    100 * break_even_cost[
        "pct_adverse_execution_per_side"
    ],
    break_even_cost["total_pnl_usd"],
    marker="o",
)
ax.axhline(0, color="black", linestyle="--")
ax.set_xlabel(
    "Adverse execution penalty per side (% of option price)"
)
ax.set_ylabel("Total P&L (USD)")
ax.set_title(
    "RQ5 ATM ML Strategy: Break-Even Execution-Cost Sensitivity"
)
ax.grid(alpha=0.2)

fig.tight_layout()
cost_fig = FIGURE_ROOT / "rq5_break_even_execution_cost.png"
fig.savefig(cost_fig, dpi=220, bbox_inches="tight")
print("Saved:", cost_fig)
plt.show()

# Part G — Hypothetical account exposure

## 14. Capital exposure for small hypothetical accounts

This is **not investment advice or a claim about a suitable account size**.

It simply expresses the observed one-contract entry premium as a percentage of several hypothetical account balances.

A strategy can be profitable historically while still requiring impractically concentrated capital exposure.

In [ ]:
account_rows = []

for offset in STRIKE_OFFSETS_POINTS:
    subset = trade_log[
        trade_log["strategy"].eq("RQ1_ML")
        & trade_log["otm_offset_points"].eq(offset)
        & trade_log["execution_scenario"].eq(
            PRIMARY_COST_SCENARIO
        )
    ].copy()

    if subset.empty:
        continue

    for account_size in HYPOTHETICAL_ACCOUNTS_USD:
        exposure = (
            subset["capital_at_risk_usd"]
            / account_size
        )

        account_rows.append(
            {
                "otm_offset_points": offset,
                "hypothetical_account_usd": account_size,
                "trades": len(subset),
                "median_trade_exposure_pct": (
                    exposure.median()
                ),
                "mean_trade_exposure_pct": (
                    exposure.mean()
                ),
                "max_trade_exposure_pct": (
                    exposure.max()
                ),
                "share_trades_affordable": (
                    subset["capital_at_risk_usd"]
                    <= account_size
                ).mean(),
                "share_trade_exposure_over_10pct": (
                    exposure > 0.10
                ).mean(),
                "share_trade_exposure_over_25pct": (
                    exposure > 0.25
                ).mean(),
                "share_trade_exposure_over_50pct": (
                    exposure > 0.50
                ).mean(),
            }
        )

account_exposure = pd.DataFrame(account_rows)

display(account_exposure)

account_exposure.to_csv(
    TABLE_ROOT / "rq5_hypothetical_account_exposure.csv",
    index=False,
)

# Part H — MFE versus MAE visual explanation

## 15. Which eventual winners first suffered large drawdowns?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

scatter = ax.scatter(
    100 * path_diagnostics["max_adverse_excursion"],
    100 * path_diagnostics["max_favorable_excursion"],
    c=100 * path_diagnostics["close_return_reference"],
    cmap="coolwarm",
    s=90,
    edgecolor="black",
    alpha=0.85,
)

ax.axvline(
    -50,
    color="darkred",
    linestyle="--",
    linewidth=1.5,
    label="−50% stop level",
)

ax.axhline(
    100,
    color="darkgreen",
    linestyle="--",
    linewidth=1.5,
    label="+100% take-profit level",
)

ax.set_xlabel("Maximum adverse excursion (%)")
ax.set_ylabel("Maximum favourable excursion (%)")
ax.set_title(
    "RQ5 ATM ML Trades: Adverse vs Favourable Excursion"
)
ax.legend()
ax.grid(alpha=0.2)

cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Reference return near close (%)")

fig.tight_layout()
mfe_mae_fig = FIGURE_ROOT / "rq5_mfe_vs_mae.png"
fig.savefig(mfe_mae_fig, dpi=220, bbox_inches="tight")
print("Saved:", mfe_mae_fig)
plt.show()

# Part I — Multi-contract take-profit + runner analysis

## 16. Multi-contract simulation assumptions

Each multi-contract strategy uses **multiple copies of the same selected option contract**.

Example:

`2C_TP100_RUNNER_CLOSE`

- enter two identical ATM contracts at the same executed entry price;
- risk two premiums, not one;
- maintain the −50% hard stop while the full position is open;
- if +100% is reached before the stop:
  - sell one contract at the +100% target reference;
  - leave one contract as a runner;
- the runner is held to the normal close exit.

For the break-even and trailing runner variants, the changed runner stop becomes active only from the **next minute bar** after the +100% take-profit. This avoids impossible intrabar ordering.

If a stop and take-profit threshold are both touched in the same minute before any scale-out, the simulation conservatively assumes the stop occurs first.

In [ ]:
def simulate_multi_contract_rule(
    contract_bars,
    rule_name,
    rule,
    execution_params,
):
    work = contract_bars.sort_values("timestamp").copy()

    entry_bar = first_entry_bar(work)
    close_bar = last_close_bar(work)

    if entry_bar is None or close_bar is None:
        return None

    entry_ref = float(entry_bar["open"])
    entry_exec = execute_buy(
        entry_ref,
        execution_params,
    )

    entry_ts = pd.Timestamp(entry_bar["timestamp"])
    close_ts = pd.Timestamp(close_bar["timestamp"])

    path = work[
        (work["timestamp"] >= entry_ts)
        & (work["timestamp"] <= close_ts)
    ].sort_values("timestamp").copy()

    if path.empty:
        return None

    total_contracts = int(rule["contracts"])
    open_contracts = total_contracts

    initial_stop_pct = rule.get("initial_stop_pct")
    active_stop = (
        entry_exec * (1 + initial_stop_pct)
        if initial_stop_pct is not None
        else None
    )

    tp1_pct = rule.get("tp1_pct")
    tp1_level = (
        entry_exec * (1 + tp1_pct)
        if tp1_pct is not None
        else None
    )

    tp2_pct = rule.get("tp2_pct")
    tp2_level = (
        entry_exec * (1 + tp2_pct)
        if tp2_pct is not None
        else None
    )

    tp1_done = False
    tp2_done = False

    runner_mode = rule.get("runner_mode", "close")
    runner_trailing_active = False
    pending_runner_stop = None
    running_high_after_tp1 = None

    ambiguous_bar = False
    realised_legs = []

    def add_exit_leg(
        timestamp,
        contracts,
        reference_price,
        reason,
    ):
        exec_price = execute_sell(
            reference_price,
            execution_params,
        )

        realised_legs.append(
            {
                "timestamp": pd.Timestamp(timestamp),
                "contracts": int(contracts),
                "reference_price": float(reference_price),
                "execution_price": float(exec_price),
                "reason": reason,
            }
        )

    for bar in path.itertuples(index=False):
        if open_contracts <= 0:
            break

        if pending_runner_stop is not None:
            active_stop = pending_runner_stop
            pending_runner_stop = None

            if runner_mode == "trail_after_tp1":
                runner_trailing_active = True

        bar_open = float(bar.open)
        bar_high = float(bar.high)
        bar_low = float(bar.low)
        ts = pd.Timestamp(bar.timestamp)

        stop_hit = (
            active_stop is not None
            and bar_low <= active_stop
        )

        tp1_hit = (
            not tp1_done
            and tp1_level is not None
            and bar_high >= tp1_level
        )

        tp2_hit = (
            not tp2_done
            and tp2_level is not None
            and bar_high >= tp2_level
        )

        # Conservative ambiguity rule before a take-profit scale-out.
        if stop_hit and (tp1_hit or tp2_hit):
            ambiguous_bar = True

            stop_ref = min(active_stop, bar_open)

            add_exit_leg(
                ts,
                open_contracts,
                stop_ref,
                "stop_before_scaleout_ambiguous",
            )

            open_contracts = 0
            break

        if stop_hit:
            stop_ref = min(active_stop, bar_open)

            add_exit_leg(
                ts,
                open_contracts,
                stop_ref,
                "stop",
            )

            open_contracts = 0
            break

        # TP1 executes first if it exists.
        if tp1_hit and open_contracts > 0:
            contracts_to_sell = min(
                int(rule["tp1_contracts"]),
                open_contracts,
            )

            if contracts_to_sell > 0:
                add_exit_leg(
                    ts,
                    contracts_to_sell,
                    tp1_level,
                    "tp1",
                )

                open_contracts -= contracts_to_sell

            tp1_done = True

            if open_contracts > 0:
                if runner_mode == "breakeven_after_tp1":
                    # Becomes active next minute.
                    pending_runner_stop = entry_exec

                elif runner_mode == "trail_after_tp1":
                    running_high_after_tp1 = max(
                        tp1_level,
                        bar_high,
                    )

                    pending_runner_stop = (
                        running_high_after_tp1
                        * (1 - rule["trail_pct"])
                    )

        # TP2 can only affect still-open contracts.
        if (
            tp2_hit
            and open_contracts > 0
        ):
            contracts_to_sell = min(
                int(rule["tp2_contracts"]),
                open_contracts,
            )

            if contracts_to_sell > 0:
                add_exit_leg(
                    ts,
                    contracts_to_sell,
                    tp2_level,
                    "tp2",
                )

                open_contracts -= contracts_to_sell

            tp2_done = True

        # Update trailing runner after TP1; new stop applies next minute.
        if (
            open_contracts > 0
            and runner_mode == "trail_after_tp1"
            and tp1_done
        ):
            if running_high_after_tp1 is None:
                running_high_after_tp1 = bar_high
            else:
                running_high_after_tp1 = max(
                    running_high_after_tp1,
                    bar_high,
                )

            pending_runner_stop = max(
                active_stop
                if active_stop is not None
                else 0.0,
                running_high_after_tp1
                * (1 - rule["trail_pct"]),
            )

    # Any remaining runner exits near close.
    if open_contracts > 0:
        add_exit_leg(
            close_ts,
            open_contracts,
            float(close_bar["close"]),
            "close_runner",
        )

        open_contracts = 0

    if not realised_legs:
        return None

    total_exited = sum(
        leg["contracts"]
        for leg in realised_legs
    )

    if total_exited != total_contracts:
        raise RuntimeError(
            f"{rule_name}: exited {total_exited} contracts "
            f"from {total_contracts}."
        )

    commission_side = float(
        execution_params["commission_per_side_usd"]
    )

    # One commission per contract on entry and one on each contract exit.
    entry_commission = (
        total_contracts * commission_side
    )

    exit_commission = (
        total_contracts * commission_side
    )

    return {
        "rule": rule_name,
        "contracts": total_contracts,
        "entry_timestamp": entry_ts,
        "entry_execution_price": entry_exec,
        "legs": realised_legs,
        "entry_commission_usd_per_position": entry_commission,
        "exit_commission_usd_per_position": exit_commission,
        "total_commission_usd_per_position": (
            entry_commission + exit_commission
        ),
        "ambiguous_intrabar": ambiguous_bar,
        "tp1_achieved": tp1_done,
        "tp2_achieved": tp2_done,
        "last_exit_timestamp": max(
            leg["timestamp"]
            for leg in realised_legs
        ),
    }

## 17. Simulate all multi-contract rules across ML ATM candidate trades

In [ ]:
multi_rows = []

for candidate in candidates.itertuples(index=False):
    date = pd.Timestamp(candidate.session_date)
    option_type = candidate.ml_direction

    selected = get_selected_contract(
        date,
        option_type,
        PRIMARY_STRIKE_OFFSET,
    )

    if selected is None:
        continue

    contract_bars = get_contract_bars(
        date,
        selected["ticker"],
    )

    multiplier_raw = pd.to_numeric(
        pd.Series(
            [selected.get("shares_per_contract", np.nan)]
        ),
        errors="coerce",
    ).iloc[0]

    multiplier = (
        float(multiplier_raw)
        if np.isfinite(multiplier_raw)
        and multiplier_raw > 0
        else DEFAULT_CONTRACT_MULTIPLIER
    )

    for rule_name, rule in MULTI_CONTRACT_RULES.items():
        result = simulate_multi_contract_rule(
            contract_bars,
            rule_name,
            rule,
            scenario_params,
        )

        if result is None:
            continue

        total_contracts = result["contracts"]

        entry_capital = (
            result["entry_execution_price"]
            * multiplier
            * total_contracts
            + result[
                "entry_commission_usd_per_position"
            ]
        )

        gross_exit_value = sum(
            leg["execution_price"]
            * multiplier
            * leg["contracts"]
            for leg in result["legs"]
        )

        entry_value = (
            result["entry_execution_price"]
            * multiplier
            * total_contracts
        )

        net_pnl = (
            gross_exit_value
            - entry_value
            - result[
                "total_commission_usd_per_position"
            ]
        )

        return_on_initial_premium = (
            net_pnl / entry_capital
            if entry_capital > 0
            else np.nan
        )

        leg_reason_counts = pd.Series(
            [
                leg["reason"]
                for leg in result["legs"]
            ]
        ).value_counts().to_dict()

        multi_rows.append(
            {
                "session_date": date,
                "rule": rule_name,
                "contracts": total_contracts,
                "option_type": option_type,
                "option_ticker": selected["ticker"],
                "entry_execution_price": (
                    result["entry_execution_price"]
                ),
                "capital_at_risk_usd": entry_capital,
                "net_pnl_usd": net_pnl,
                "return_on_initial_premium": (
                    return_on_initial_premium
                ),
                "winner": net_pnl > 0,
                "tp1_achieved": result["tp1_achieved"],
                "tp2_achieved": result["tp2_achieved"],
                "ambiguous_intrabar": (
                    result["ambiguous_intrabar"]
                ),
                "last_exit_timestamp": (
                    result["last_exit_timestamp"]
                ),
                "tp1_contracts_exited": (
                    leg_reason_counts.get("tp1", 0)
                ),
                "tp2_contracts_exited": (
                    leg_reason_counts.get("tp2", 0)
                ),
                "stop_contracts_exited": sum(
                    count
                    for reason, count
                    in leg_reason_counts.items()
                    if "stop" in reason
                ),
                "runner_contracts_exited_at_close": (
                    leg_reason_counts.get(
                        "close_runner",
                        0,
                    )
                ),
                "legs_json": json.dumps(
                    [
                        {
                            **leg,
                            "timestamp": (
                                leg["timestamp"].isoformat()
                            ),
                        }
                        for leg in result["legs"]
                    ]
                ),
            }
        )

multi_trades = pd.DataFrame(multi_rows)

display(multi_trades.head(30))

multi_trades.to_parquet(
    ROBUST_ROOT / "rq5_multi_contract_trade_log.parquet",
    index=False,
)
multi_trades.to_csv(
    ROBUST_ROOT / "rq5_multi_contract_trade_log.csv",
    index=False,
)

## 18. Multi-contract results: total dollars versus capital efficiency

In [ ]:
multi_summary_rows = []

for rule, subset in multi_trades.groupby("rule"):
    equity = subset.sort_values("session_date").copy()
    equity["cum_pnl"] = equity["net_pnl_usd"].cumsum()
    equity["peak"] = equity["cum_pnl"].cummax()
    equity["drawdown"] = equity["cum_pnl"] - equity["peak"]

    multi_summary_rows.append(
        {
            "rule": rule,
            "contracts": int(subset["contracts"].iloc[0]),
            "trades": len(subset),
            "total_pnl_usd": subset["net_pnl_usd"].sum(),
            "mean_pnl_usd": subset["net_pnl_usd"].mean(),
            "median_pnl_usd": subset["net_pnl_usd"].median(),
            "win_rate": subset["winner"].mean(),
            "mean_return_on_initial_premium": (
                subset[
                    "return_on_initial_premium"
                ].mean()
            ),
            "median_return_on_initial_premium": (
                subset[
                    "return_on_initial_premium"
                ].median()
            ),
            "median_capital_at_risk_usd": (
                subset["capital_at_risk_usd"].median()
            ),
            "mean_capital_at_risk_usd": (
                subset["capital_at_risk_usd"].mean()
            ),
            "profit_factor": profit_factor(
                subset["net_pnl_usd"]
            ),
            "max_drawdown_usd": (
                equity["drawdown"].min()
            ),
            "tp1_trade_share": (
                subset["tp1_achieved"].mean()
            ),
            "ambiguous_bar_share": (
                subset["ambiguous_intrabar"].mean()
            ),
        }
    )

multi_summary = pd.DataFrame(
    multi_summary_rows
).sort_values("rule")

display(multi_summary)

multi_summary.to_csv(
    TABLE_ROOT / "rq5_multi_contract_summary.csv",
    index=False,
)

## 19. Incremental value of the runner

Raw total P&L is not enough because a two-contract strategy naturally commits roughly twice the capital of a one-contract strategy.

This table therefore compares:

- total dollars earned;
- return on total initial premium;
- median capital required;
- drawdown;
- incremental P&L relative to simply buying the equivalent number of contracts and holding all of them to close.

The last comparison asks whether the **scale-out logic itself** added value, rather than merely benefiting from larger position size.

In [ ]:
hold1 = multi_trades[
    multi_trades["rule"].eq("1C_HOLD_CLOSE")
][
    [
        "session_date",
        "net_pnl_usd",
        "capital_at_risk_usd",
    ]
].rename(
    columns={
        "net_pnl_usd": "one_contract_hold_pnl",
        "capital_at_risk_usd": "one_contract_hold_capital",
    }
)

incremental_rows = []

for rule, subset in multi_trades.groupby("rule"):
    contracts = int(subset["contracts"].iloc[0])

    merged = subset.merge(
        hold1,
        on="session_date",
        how="inner",
        validate="one_to_one",
    )

    # Counterfactual: buy the same number of contracts and hold all to close.
    merged[
        "same_size_all_hold_pnl"
    ] = (
        merged["one_contract_hold_pnl"]
        * contracts
    )

    merged[
        "rule_minus_same_size_hold_pnl"
    ] = (
        merged["net_pnl_usd"]
        - merged["same_size_all_hold_pnl"]
    )

    incremental_rows.append(
        {
            "rule": rule,
            "contracts": contracts,
            "paired_trades": len(merged),
            "actual_total_pnl_usd": (
                merged["net_pnl_usd"].sum()
            ),
            "same_size_all_hold_total_pnl_usd": (
                merged[
                    "same_size_all_hold_pnl"
                ].sum()
            ),
            "incremental_pnl_from_exit_logic_usd": (
                merged[
                    "rule_minus_same_size_hold_pnl"
                ].sum()
            ),
            "mean_incremental_pnl_per_trade_usd": (
                merged[
                    "rule_minus_same_size_hold_pnl"
                ].mean()
            ),
            "share_rule_beats_same_size_hold": (
                merged[
                    "rule_minus_same_size_hold_pnl"
                ] > 0
            ).mean(),
        }
    )

multi_incremental = pd.DataFrame(incremental_rows)

display(multi_incremental)

multi_incremental.to_csv(
    TABLE_ROOT / "rq5_multi_contract_incremental_vs_same_size_hold.csv",
    index=False,
)

## 20. Multi-contract hypothetical account exposure

In [ ]:
multi_account_rows = []

for rule, subset in multi_trades.groupby("rule"):
    for account in HYPOTHETICAL_ACCOUNTS_USD:
        exposure = (
            subset["capital_at_risk_usd"]
            / account
        )

        multi_account_rows.append(
            {
                "rule": rule,
                "contracts": int(
                    subset["contracts"].iloc[0]
                ),
                "hypothetical_account_usd": account,
                "median_position_exposure_pct": (
                    exposure.median()
                ),
                "mean_position_exposure_pct": (
                    exposure.mean()
                ),
                "max_position_exposure_pct": (
                    exposure.max()
                ),
                "share_positions_affordable": (
                    subset["capital_at_risk_usd"]
                    <= account
                ).mean(),
                "share_positions_over_25pct_account": (
                    exposure > 0.25
                ).mean(),
                "share_positions_over_50pct_account": (
                    exposure > 0.50
                ).mean(),
            }
        )

multi_account_exposure = pd.DataFrame(
    multi_account_rows
)

display(multi_account_exposure)

multi_account_exposure.to_csv(
    TABLE_ROOT / "rq5_multi_contract_account_exposure.csv",
    index=False,
)

# Part J — Conservative interpretation generator

## 21. Generate a dissertation-oriented robustness summary

In [ ]:
all_row = profit_concentration[
    profit_concentration["removed_top_winners"].eq(0)
].iloc[0]

remove1 = profit_concentration[
    profit_concentration["removed_top_winners"].eq(1)
]

primary_multi = multi_summary[
    multi_summary["rule"].eq(
        PRIMARY_MULTI_CONTRACT_RULE
    )
]

primary_incremental = multi_incremental[
    multi_incremental["rule"].eq(
        PRIMARY_MULTI_CONTRACT_RULE
    )
]

top1_share = top_contribution.loc[
    top_contribution["top_n_trades"].eq(1),
    "share_of_net_total_pnl",
]

top1_share_text = (
    f"{top1_share.iloc[0]:.1%}"
    if len(top1_share)
    else "not available"
)

if len(stopped):
    recovered_entry_share = (
        stopped[
            "after_stop_recovered_to_entry"
        ].mean()
    )
    later_tp100_share = (
        stopped[
            "after_stop_reached_plus100"
        ].mean()
    )
else:
    recovered_entry_share = np.nan
    later_tp100_share = np.nan

if len(primary_multi):
    pm = primary_multi.iloc[0]
    pm_text = (
        f"The pre-specified two-contract scale-out rule generated "
        f"total P&L of ${pm['total_pnl_usd']:.2f} across "
        f"{int(pm['trades'])} opportunities, with mean return on total "
        f"initial premium of "
        f"{pm['mean_return_on_initial_premium']:.1%} and median capital "
        f"requirement of ${pm['median_capital_at_risk_usd']:.2f}."
    )
else:
    pm_text = (
        "The primary two-contract scale-out rule could not be evaluated."
    )

if len(primary_incremental):
    pi = primary_incremental.iloc[0]
    if pi["incremental_pnl_from_exit_logic_usd"] > 0:
        scaleout_text = (
            f"Compared with simply buying two contracts and holding both "
            f"to close, the partial-profit/runner logic added "
            f"${pi['incremental_pnl_from_exit_logic_usd']:.2f} in total "
            "historical P&L."
        )
    else:
        scaleout_text = (
            f"Compared with simply buying two contracts and holding both "
            f"to close, the partial-profit/runner logic reduced total "
            f"historical P&L by "
            f"${abs(pi['incremental_pnl_from_exit_logic_usd']):.2f}. "
            "This indicates that any improvement in raw dollar P&L from "
            "holding multiple contracts should not be confused with value "
            "added by the exit rule itself."
        )
else:
    scaleout_text = ""

draft = f"""
### RQ5 Strategy Robustness and Multi-Contract Sensitivity

The primary ATM ML strategy produced total historical P&L of
${all_row['total_pnl_usd']:.2f}. Profit-concentration analysis showed that
the single largest trade accounted for approximately {top1_share_text} of
net total P&L. Leave-one-trade-out and top-winner-removal tests were therefore
used to assess whether the result remained positive when unusually favourable
observations were excluded.

Path analysis also demonstrated why fixed stop-loss and take-profit rules can
materially change 0DTE outcomes. Among trades that touched the −50% level,
approximately {recovered_entry_share:.1%} later recovered to the original
entry premium and {later_tp100_share:.1%} subsequently reached +100%.
Consequently, a hard stop can remove some positions that would later become
large winners. Conversely, the post-+100% analysis measures how much additional
upside remained after a conventional doubling-of-premium take-profit level had
already been reached.

The time-of-hour analysis was used to identify when the economic value of the
selected opportunities emerged between 15:00 and the close, while the
same-date random-direction Monte Carlo separated the value of directional
prediction from the value of the opportunity-selection filter.

{pm_text} {scaleout_text}

The multi-contract analysis should be interpreted cautiously. Buying two or
three SPX contracts increases the required premium approximately in proportion
to position size, meaning a strategy may earn more dollars while becoming less
appropriate for a small retail account. For this reason, the study reports both
dollar P&L and return on total initial premium, together with hypothetical
account-exposure measures.

Overall, these robustness tests are intended to explain the source and
fragility of the RQ5 backtest rather than to optimise a new strategy on the
same small historical sample.
"""

print(draft)

draft_path = ROBUST_ROOT / "rq5_strategy_robustness_draft.md"
draft_path.write_text(draft, encoding="utf-8")
print("Saved:", draft_path)

## 22. Save manifest and output inventory

In [ ]:
manifest = {
    "research_question": "RQ5",
    "analysis": "strategy_robustness_and_multi_contract",
    "fresh_holdout_used": False,
    "primary_strategy_preserved": True,
    "primary_strike_offset_points": PRIMARY_STRIKE_OFFSET,
    "primary_execution_scenario": PRIMARY_COST_SCENARIO,
    "primary_multi_contract_sensitivity": PRIMARY_MULTI_CONTRACT_RULE,
    "multi_contract_rules": MULTI_CONTRACT_RULES,
    "time_checkpoints": TIME_CHECKPOINTS,
    "random_direction_simulations": N_RANDOM_DIRECTION_SIMULATIONS,
    "hypothetical_accounts_usd": HYPOTHETICAL_ACCOUNTS_USD,
    "break_even_cost_grid": {
        "start": float(BREAK_EVEN_PCT_GRID.min()),
        "stop": float(BREAK_EVEN_PCT_GRID.max()),
        "step": 0.005,
        "commission_per_side_usd": (
            BREAK_EVEN_COMMISSION_PER_SIDE_USD
        ),
    },
    "methodology_notes": [
        "Multi-contract strategies use multiple copies of the same option contract.",
        "Raw multi-contract dollar P&L is not treated as evidence of better capital efficiency.",
        "Each multi-contract rule is compared with an equal-number-of-contracts hold-to-close counterfactual.",
        "If stop and take-profit are touched in the same one-minute bar before scale-out, stop is assumed first.",
        "Runner break-even/trailing changes become active on the next minute.",
        "Same-date random-direction Monte Carlo tests direction value, not day-selection value.",
        "A full random-day benchmark requires new option retrieval on non-candidate sessions.",
        "Hypothetical account-size outputs are descriptive exposure analysis, not personalised investment advice.",
    ],
}

manifest_path = (
    ROBUST_ROOT / "rq5_strategy_robustness_manifest.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

inventory = [
    "rq5_profit_concentration.csv",
    "rq5_top_winner_contribution.csv",
    "rq5_leave_one_trade_out.csv",
    "rq5_stop_tp_path_recovery.csv",
    "rq5_stop50_recovery_summary.csv",
    "rq5_post_tp100_upside_summary.csv",
    "rq5_time_of_hour_development.csv",
    "rq5_call_put_decomposition.csv",
    "rq5_random_direction_monte_carlo.csv",
    "rq5_break_even_execution_cost.csv",
    "rq5_hypothetical_account_exposure.csv",
    "rq5_multi_contract_summary.csv",
    "rq5_multi_contract_incremental_vs_same_size_hold.csv",
    "rq5_multi_contract_account_exposure.csv",
]

print("Manifest:", manifest_path)
print("\nImportant table outputs:")
for name in inventory:
    print(" -", TABLE_ROOT / name)

print("\nRobustness artifacts:")
print(" -", ROBUST_ROOT / "rq5_multi_contract_trade_log.parquet")
print(" -", ROBUST_ROOT / "rq5_strategy_robustness_draft.md")

print("\nFigures:")
for name in [
    "rq5_time_of_hour_development.png",
    "rq5_random_direction_monte_carlo.png",
    "rq5_break_even_execution_cost.png",
    "rq5_mfe_vs_mae.png",
]:
    print(" -", FIGURE_ROOT / name)

# 23. How to interpret the multi-contract section

The crucial comparison is **not**:

> "Did two contracts make more dollars than one contract?"

Two contracts should usually create larger dollar gains *and* larger dollar losses because twice as much premium is being risked.

Instead compare:

### A. Capital efficiency

`mean_return_on_initial_premium`

This asks how much was earned relative to the total premium committed to the full multi-contract position.

### B. Exit-rule value

`incremental_pnl_from_exit_logic_usd`

For a two-contract rule, the notebook compares it with the counterfactual:

> buy two identical contracts and hold both to close.

If the scale-out strategy makes $15,000 but simply holding two contracts would have made $19,000, the multiple-contract strategy increased raw dollars only because more capital was deployed; the partial take-profit actually **reduced** economic value.

### C. Risk and accessibility

Use:

- median total capital required;
- maximum drawdown;
- hypothetical account exposure;
- percentage of positions requiring >25% or >50% of a hypothetical account.

This is especially important when discussing viability for a small retail participant.

### Primary multi-contract question

The pre-specified test is:

> **Does selling one of two contracts at +100% and leaving one runner improve the risk/return trade-off relative to holding the position unchanged?**

The break-even and trailing runner variants help explain sensitivity but should not replace the primary rule simply because one happens to perform best historically.